<a href="https://colab.research.google.com/github/einfachstiffi-boop/Project-app/blob/main/Machine_Learning_Model_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install mlcroissant

In [ ]:
import mlcroissant as mlc
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score


In [ ]:


# Fetch the Croissant JSON-LD
croissant_dataset = mlc.Dataset('https://www.kaggle.com/datasets/solomonameh/spotify-music-dataset/croissant/download')

# Check what record sets are in the dataset
record_sets = croissant_dataset.metadata.record_sets
print(record_sets)

# Fetch the records and put them in a DataFrame
record_set_df = pd.DataFrame(croissant_dataset.records(record_set=record_sets[0].uuid))
record_set_df.head()


In [ ]:
for col in record_set_df.columns:
    if record_set_df[col].dtype == object:
        record_set_df[col] = record_set_df[col].apply(
            lambda x: x.decode("utf8") if isinstance(x, bytes) else x
        )



record_set_df.columns = [
    name.split("/")[-1] for name in record_set_df.columns
]

record_set_df=pd.DataFrame(record_set_df)
record_set_df



In [ ]:
#all genres in the data
record_set_df["playlist_genre"].unique()

In [ ]:
#all min max values in the columns
numeric_df = record_set_df.select_dtypes(include=["number"])

min_max = pd.DataFrame({
    "min": numeric_df.min(),
    "max": numeric_df.max()
})

min_max



In [ ]:
#each bin represents a potential listener category
bin1 = [
     'afrobeats', 'arabic', 'brazilian', 'gaming', 'hip-hop', 'k-pop', 'latin', 'pop', 'r&b', 'reggae'
]

bin2 = [
    'ambient', 'blues', 'folk', 'indian', 'indie', 'korean', 'soul'
]

bin3 = [
    'classical', 'jazz', 'lofi'
]

bin4 = [
    'country', 'electronic', 'j-pop', 'metal', 'punk', 'rock', 'turkish', 'world'
]


def assign_bin(g):
    if g in bin1:
        return "bin1"
    if g in bin2:
        return "bin2"
    if g in bin3:
        return "bin3"
    return "bin4"

record_set_df["genre_bin"] = record_set_df["playlist_genre"].apply(assign_bin)

record_set_df

In [ ]:
#droping column we do not need
cols_to_drop = [
    "playlist_id",
    "type",
    "track_album_name",
    "playlist_name",
    "analysis_url",
    "track_id",
    "track_name",
    "track_album_release_date",
    "track_album_id",
    "mode",
    "key",
    'track_artist',
    "track_href",
"id",
    "playlist_genre",
    "uri",
    "valence",
    'liveness',
    "instrumentalness"
]

record_set_del = record_set_df.drop(columns=cols_to_drop)


In [ ]:
record_set_del

In [ ]:
X = record_set_del.drop(columns=["genre_bin"])
y = record_set_del["genre_bin"]

#Encode the categorical column:
X = pd.get_dummies(X, columns=["playlist_subgenre"], drop_first=True)




In [ ]:
features = [
    "energy",
    "tempo",
    "danceability",
    "loudness",
    "time_signature",
    "speechiness",
    "track_popularity",
    "acousticness"
]

X = record_set_df[features]
y = record_set_df["genre_bin"]

from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=300, random_state=42)
model.fit(X, y)

import joblib
joblib.dump(model, "model.pkl")

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Random Forest Modell trainieren

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))



In [ ]:
y_test = y_test.values.reshape(-1)

print(y_test.shape)
print(y_pred.shape)
